# Predicción de Abandono Temprano en Spotify — TP Minería de Datos (UTDT)

**Notebook de la Clase 2** (validación formal + árboles de decisión), copia de `tp_spotify_clase1_knn.ipynb` con el contenido de la Clase 1 intacto (EDA, feature engineering, baseline KNN) más la sección nueva de la Clase 2 al final. Cada clase queda en su propio archivo como registro de estudio; el consolidado final para la entrega se arma más adelante, cerca de la fecha límite.

## Base heredada de la Clase 1 (mínima)

Cada clase vive en su propio notebook. El razonamiento completo (EDA con 2 figuras, comparación de tres conjuntos de features, barrido fino de K, chequeo temporal) queda en `tp_spotify_clase1_knn.ipynb` — no se repite acá, solo se usa el resultado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
DATA_DIR = "competition_data"

# Estilo único para todos los gráficos del notebook.
# Paleta categórica validada para daltonismo (orden fijo: azul, aqua, amarillo).
C_AZUL, C_AQUA, C_AMARILLO = "#2a78d6", "#1baf7a", "#eda100"
C_TEXTO, C_MUTED, C_GRILLA, C_EJES = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": C_GRILLA, "grid.linewidth": 0.8,
    "axes.edgecolor": C_EJES, "axes.labelcolor": C_TEXTO,
    "xtick.color": C_MUTED, "ytick.color": C_MUTED,
})

In [ ]:
USECOLS = [
    "obs_id", "ms_played", "ts", "platform_family", "conn_country",
    "master_metadata_album_artist_name",
    "spotify_track_uri", "spotify_episode_uri", "audiobook_uri",
    "shuffle", "offline", "incognito_mode",
    "user_id",  # excluida en la Clase 1 como "identificador"; decisión revisada en la sección final de este notebook
]
DTYPES = {
    "spotify_track_uri": "string",
    "spotify_episode_uri": "string",
    "audiobook_uri": "string",
}

train_raw = pd.read_csv(f"{DATA_DIR}/train_data.txt", sep="\t", usecols=USECOLS, dtype=DTYPES)

train_raw["target"] = (train_raw["ms_played"] < 30000).astype(int)

print("Filas:", len(train_raw))
print("Proporción de abandono temprano (target=1):", train_raw["target"].mean().round(4))
train_raw[["obs_id", "ms_played", "target"]].head()

In [ ]:
# Mismas tres derivadas de la Clase 1 (justificación completa en esa clase, acá solo el resultado):
# content_type (qué URI está presente), hora/dia_semana (de ts, en UTC), artista_top (top 50 de train, resto -> "otros").

train_raw["content_type"] = np.select(
    [train_raw["spotify_track_uri"].notna(),
     train_raw["spotify_episode_uri"].notna(),
     train_raw["audiobook_uri"].notna()],
    ["track", "episode", "audiobook"],
    default="desconocido",
)

ts_dt = pd.to_datetime(train_raw["ts"], utc=True, format="ISO8601")
train_raw["ts_dt"] = ts_dt
train_raw["hora"] = ts_dt.dt.hour
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
train_raw["dia_semana"] = ts_dt.dt.dayofweek.map(dict(enumerate(DIAS)))

col_artista = "master_metadata_album_artist_name"
TOP_ARTISTAS = train_raw[col_artista].value_counts().head(50).index
train_raw["artista_top"] = train_raw[col_artista].where(
    train_raw[col_artista].isin(TOP_ARTISTAS), "otros"
)

print("Features derivadas listas. artista_top:", train_raw["artista_top"].nunique(), "categorías (incluye 'otros').")

In [ ]:
# La Clase 1 comparó 3 conjuntos anidados con el mismo holdout y grilla de K; ganó el conjunto C
# (todas las variables: contexto + momento + contenido), con AUC holdout aleatorio 0.6903 (K=101)
# que bajaba a 0.6477 al respetar el tiempo. Partimos directo de ese resultado ya validado.
TODAS_LAS_FEATURES = [
    "platform_family", "conn_country", "shuffle", "offline", "incognito_mode",
    "hora", "dia_semana", "content_type", "artista_top",
]
mejor_feats = TODAS_LAS_FEATURES

## Clase 2 — Validación formal (train/validation/test) y Árboles de Decisión

Todo lo de arriba es la base mínima heredada de la Clase 1 (`train_raw` con target y features derivadas, `TODAS_LAS_FEATURES`/`mejor_feats` = conjunto C ya validado). De acá en adelante es contenido nuevo de la Clase 2:

1. Un **esquema de validación formal train / validation / test**, que reemplaza el holdout único (y el "chequeo de honestidad" improvisado) de la Clase 1.
2. **Árboles de decisión** como segundo algoritmo, diseñados con lo visto en clase y comparados de forma honesta contra un campeón de KNN.

### Esquema de validación: test interno + CV temporal de ventana expansiva

En la Clase 1 alcanzaba con un único holdout: una sola decisión (el K). Acá vamos a tomar *muchas*: profundidad, mínimos por hoja y por split, poda por impureza (α), ventana de datos, y encima comparar dos familias de algoritmos (árbol vs. KNN). La clase lo nombró con precisión — **"overfitting the validation set"** (Andrew Ng): si mirás el mismo conjunto una y otra vez mientras probás combinaciones, tarde o temprano alguna le pega de casualidad al ruido y el número queda inflado.

El esquema, adaptado a datos **no i.i.d.** (el profesor fue explícito: el TP tiene componente temporal, los gustos cambian):

- **`train` (85%, cronológicamente el pasado):** acá se toman TODAS las decisiones — pero no contra un validation fijo, sino con **CV temporal de ventana expansiva**, la receta explícita de la clase para series de tiempo: entrenar con el pasado, validar con el futuro inmediato, "de a ventanas", y promediar. Promediar varias ventanas da una señal menos ruidosa que un holdout único → elegir el `argmax` es más confiable. *(Una versión anterior de este notebook usaba train/validation/test 70/15/15 con validation cronológico fijo: ese 15% quedaba ocioso durante la selección y su número único era más ruidoso — esta versión lo reemplaza por las ventanas y aprovecha esas filas como desarrollo.)*
- **`test_interno` (15% final, hasta 2024-08-31):** no se toca hasta el final, una única vez, con los modelos ya elegidos — estimación honesta, mismo rol que el leaderboard público de Kaggle pero sin gastar submits.
- **K-Fold aleatorio: no.** El profesor lo descarta para datasets grandes en contexto i.i.d., y además acá mezclar épocas está prohibido: **split cronológico siempre**, en el corte train/test_interno y dentro de cada ventana de la CV.

In [ ]:
train_raw_ord = train_raw.sort_values("ts_dt").reset_index(drop=True)

n = len(train_raw_ord)
n_train = int(n * 0.85)
# train (desarrollo): todas las decisiones se toman acá, vía CV temporal interna.
# test_interno: el 15% más reciente, una sola mirada al final.

df_train = train_raw_ord.iloc[:n_train]
df_test_interno = train_raw_ord.iloc[n_train:]

def build_xy(df, features=None):
    features = TODAS_LAS_FEATURES if features is None else features
    X = df[features].fillna("missing").astype(str)
    y = df["target"]
    return X, y

X_te, y_te = build_xy(df_test_interno)

for nombre, df in [("train", df_train), ("test interno", df_test_interno)]:
    print(f"{nombre:<13} {len(df):>7} filas | {df['ts_dt'].min().date()} -> {df['ts_dt'].max().date()} "
          f"| tasa abandono {df['target'].mean():.4f}")

**Dos observaciones honestas sobre estos números:**

- `test_interno` termina el **2024-08-31**, exactamente donde termina `train_data.txt` según la consigna (test real de Kaggle: desde el 2024-09-01). No es casualidad — es la porción más reciente de nuestros datos, la más parecida en el tiempo al test real, así que es el mejor proxy interno que podemos construir.
- La tasa de abandono en `test_interno` (≈0.230) es notablemente más baja que en `train` (≈0.287). Esto es exactamente lo que un split aleatorio (i.i.d.) escondería mezclando épocas, y que un split cronológico deja a la vista: el comportamiento de escucha cambió con el tiempo. Razón de más para validar respetando la cronología, como advirtió la clase.

### Árbol de decisión: por qué el pipeline es más simple que el de KNN

Un árbol parte el espacio de atributos con cortes del tipo "¿esta variable es mayor o menor a tanto?", anidados desde la raíz hacia las hojas. En clasificación, cada hoja predice la **proporción de cada clase** entre las observaciones de entrenamiento que caen ahí — directamente una probabilidad, que es lo que necesitamos para ROC-AUC.

Dos diferencias con el pipeline de KNN de la Clase 1:

1. **Sin `StandardScaler`.** KNN necesitaba escalar porque la distancia euclídea se deja dominar por la variable de mayor varianza. Un árbol no calcula distancias: cada corte es un umbral sobre una única variable a la vez, y ese umbral separa las mismas observaciones esté la variable en la escala que esté. Escalar no cambia ni un solo corte que el árbol pueda elegir.
2. **Sigue haciendo falta `OneHotEncoder`.** La clase mostró que un árbol *podría*, en teoría, partir directo por variable categórica (una rama por categoría) — pero scikit-learn no lo tiene implementado así. Seguimos con one-hot, que la propia clase aclaró que "no es tan dramático" (ni siquiera XGBoost lo tuvo durante años y aun así competía bien).

Y a diferencia de KNN (*lazy*: todo el costo está en `.predict()`), un árbol es ***eager***: todo el costo de cómputo está en `.fit()` (construir el árbol); predecir es solo recorrer ifs, casi gratis. Por eso ahora entrenamos con **todas** las filas de `train` (~638.000), sin subsamplear — el motivo por el que subsampleábamos en la Clase 1 era exclusivo de KNN.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def build_tree_pipeline(features, **tree_kwargs):
    preprocessor = ColumnTransformer(
        transformers=[
            ("onehot", OneHotEncoder(handle_unknown="ignore"), features),
        ],
        remainder="drop",
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("tree", DecisionTreeClassifier(random_state=RANDOM_STATE, **tree_kwargs)),
        ]
    )

### Herramienta de validación: CV temporal de ventana expansiva

Antes de elegir nada, definimos la pieza que la clase prescribe para datos con estructura temporal. `make_expanding_folds` parte `df_train` (ya ordenado por fecha) en `n_folds+1` bloques cronológicos y arma folds *expansivos*: el fold *i* entrena con todos los bloques anteriores y valida con el siguiente. Así cada validación mira **el futuro inmediato** del entrenamiento — igual que el uso real del modelo — y ningún dato futuro se filtra hacia atrás.

`cv_temporal_auc` corre esos folds para un pipeline dado y devuelve el **AUC promedio** (y el de cada fold). Ese promedio es la brújula que reemplaza al holdout único: menos ruidoso, y por lo tanto elegir el `argmax` sobre él es mucho más confiable. Parámetros opcionales `subsample`/`val_subsample` existen solo para que KNN (caro al predecir) sea factible; el árbol no los necesita. `filtrar_reciente` se usa más abajo para el experimento de datos rancios.

In [ ]:
from itertools import product

def make_expanding_folds(df, n_folds=4):
    """Folds temporales de ventana expansiva. df debe venir ordenado cronológicamente.
    Divide en n_folds+1 bloques; el fold i entrena con [0..i] y valida con el bloque i+1."""
    n = len(df)
    bordes = np.linspace(0, n, n_folds + 2, dtype=int)  # n_folds+1 bloques
    folds = []
    for i in range(1, n_folds + 1):
        idx_train = np.arange(0, bordes[i])
        idx_val = np.arange(bordes[i], bordes[i + 1])
        if len(idx_val) > 0 and len(idx_train) > 0:
            folds.append((idx_train, idx_val))
    return folds

def cv_temporal_auc(df, features, pipeline_factory, n_folds=4,
                    subsample=None, val_subsample=None,
                    train_filter=None, random_state=RANDOM_STATE):
    """AUC promedio en validación con ventana expansiva temporal.
    df ordenado por tiempo; pipeline_factory(features) -> pipeline sin fit.
    subsample/val_subsample: topes de tamaño (para que KNN sea factible).
    train_filter: función opcional aplicada al train de cada fold (p. ej. quedarse con meses recientes)."""
    aucs = []
    for idx_tr, idx_val in make_expanding_folds(df, n_folds):
        df_tr = df.iloc[idx_tr]
        df_va = df.iloc[idx_val]
        if train_filter is not None:
            df_tr = train_filter(df_tr)
        if subsample is not None and len(df_tr) > subsample:
            df_tr = df_tr.sample(subsample, random_state=random_state)
        if val_subsample is not None and len(df_va) > val_subsample:
            df_va = df_va.sample(val_subsample, random_state=random_state)
        X_tr_f, y_tr_f = build_xy(df_tr, features)
        X_va_f, y_va_f = build_xy(df_va, features)
        pipe = pipeline_factory(features)
        pipe.fit(X_tr_f, y_tr_f)
        aucs.append(roc_auc_score(y_va_f, pipe.predict_proba(X_va_f)[:, 1]))
    return float(np.mean(aucs)), aucs

def filtrar_reciente(df, meses):
    """Se queda con las filas de los últimos `meses` meses, relativo al máximo ts del propio df.
    meses=None -> toda la historia."""
    if meses is None:
        return df
    corte = df["ts_dt"].max() - pd.DateOffset(months=int(meses))
    return df[df["ts_dt"] >= corte]

### Eligiendo los hiperparámetros con una grilla CONJUNTA y CV temporal

Dos cambios respecto del enfoque anterior (holdout único + un hiperparámetro a la vez):

1. **Grilla conjunta, no secuencial.** La clase tuneó *varios* hiperparámetros a la vez (su código de ejemplo barrió 175 combinaciones) porque los hiperparámetros **interactúan** — se ve clarísimo en que `min_samples_leaf` importa mucho más a `max_depth` alto. Adoptar de a uno se queda en óptimos locales. Barremos entonces el producto de:
   - `max_depth`: niveles de cortes (poca → underfitting; mucha → overfitting).
   - `min_samples_leaf`: mínimo de observaciones por hoja (freno directo al overfitting).
   - `min_samples_split`: mínimo para siquiera intentar partir un nodo (la clase también lo tuneó; antes lo dejábamos en default).
   - `min_impurity_decrease` (el **α** de la clase): solo acepta un split si baja la impureza al menos α — "el precio que pagás por un split". Es otro control de flexibilidad, al mismo nivel que la profundidad, así que va *dentro* de la grilla.

   `criterion` (gini vs. entropía) queda **fuera del barrido, fijo en gini, por cómputo**: la clase presentó los dos como casi equivalentes, así que es el eje que menos información pierde al recortarse. Si sobra tiempo antes de la entrega, se repone en una corrida final.

2. **Selección por CV temporal, no por un holdout único.** Cada combinación se evalúa con el AUC promedio de las ventanas expansivas. El profesor fue textual: *"nunca hay regla de dedo, la regla es probar"* — probamos, pero sobre una señal menos ruidosa. `test_interno` no se toca todavía.

In [ ]:
# Grilla CONJUNTA (la clase tuneó varios hiperparámetros a la vez, no de a uno).
# criterion queda fijo en gini (casi equivalente a entropía según la clase) para recortar cómputo.
param_grid = {
    "max_depth":             [4, 6, 8, 10, 14, 20],
    "min_samples_leaf":      [1, 20, 100],
    "min_samples_split":     [2, 50],
    "min_impurity_decrease": [0.0, 1e-4],
}
N_FOLDS = 4  # ventanas expansivas temporales dentro de df_train

combos = [dict(zip(param_grid, vals)) for vals in product(*param_grid.values())]
print(f"Combinaciones a evaluar: {len(combos)}  (x {N_FOLDS} folds temporales c/u)")

resultados = []
for params in combos:
    factory = lambda feats, p=params: build_tree_pipeline(feats, **p)
    auc_mean, auc_folds = cv_temporal_auc(df_train, mejor_feats, factory, n_folds=N_FOLDS)
    resultados.append({**params, "auc_cv": auc_mean, "auc_folds": auc_folds})

res_df = pd.DataFrame(resultados).sort_values("auc_cv", ascending=False).reset_index(drop=True)
mejor_params = {k: res_df.loc[0, k] for k in param_grid}
mejor_params["max_depth"] = int(mejor_params["max_depth"])
mejor_params["min_samples_leaf"] = int(mejor_params["min_samples_leaf"])
mejor_params["min_samples_split"] = int(mejor_params["min_samples_split"])
mejor_params["min_impurity_decrease"] = float(mejor_params["min_impurity_decrease"])
mejor_auc_arbol = float(res_df.loc[0, "auc_cv"])

# alias por compatibilidad con menciones sueltas
mejor_depth = mejor_params["max_depth"]
mejor_leaf = mejor_params["min_samples_leaf"]

print("\nMejores hiperparámetros (CV temporal):")
for k, v in mejor_params.items():
    print(f"  {k}: {v}")
print(f"AUC CV temporal (promedio de {N_FOLDS} ventanas): {mejor_auc_arbol:.4f}")
res_df.drop(columns="auc_folds").head(10)

In [ ]:
# Visualización: fijando el resto de hiperparámetros en su mejor valor, AUC CV vs profundidad por hoja.
fijos = {k: mejor_params[k] for k in ["min_samples_split", "min_impurity_decrease"]}
sub = res_df[(res_df["min_samples_split"] == fijos["min_samples_split"]) &
             (res_df["min_impurity_decrease"] == fijos["min_impurity_decrease"])]

plt.figure(figsize=(7, 4.5))
for leaf, color in zip(sorted(param_grid["min_samples_leaf"]), [C_AZUL, C_AQUA, C_AMARILLO]):
    s = sub[sub["min_samples_leaf"] == leaf].sort_values("max_depth")
    plt.plot(s["max_depth"], s["auc_cv"], marker="o", color=color, label=f"min_samples_leaf={leaf}")
plt.xlabel("max_depth")
plt.ylabel("ROC-AUC (CV temporal, promedio de ventanas)")
plt.title("Árbol: AUC de CV temporal según profundidad y min_samples_leaf")
plt.legend(frameon=False,
           title=f"split={fijos['min_samples_split']}, α={fijos['min_impurity_decrease']}, gini")
plt.show()

**Para interpretar el gráfico:** ahora cada punto es el **promedio de varias ventanas temporales**, no un único holdout, así que la curva debería salir bastante más estable que el zigzag que teníamos antes (ese zigzag era, justamente, el ruido de mirar un solo corte). Lo que esperamos leer con nitidez es el patrón que enseñó la clase: `max_depth` chico hace *underfitting* y `max_depth` grande sin freno (`min_samples_leaf=1`, α=0) hace *overfitting*, con las curvas de `min_samples_leaf` alto sosteniéndose mejor en profundidades grandes.

Elegimos la combinación que **maximiza el AUC de la CV temporal**. Un criterio extra, consistente con la clase: si dos combinaciones quedan dentro del ruido entre folds (mirá la dispersión de `auc_folds`), preferimos la **más rígida** (menor profundidad / mayor `min_samples_leaf` / mayor α) — la clase insistió en que, ante empate, menos flexibilidad es la apuesta segura contra el overfitting, y además reduce el riesgo de haber agarrado un pico de suerte.

### Decisión de datos: ¿toda la historia o solo lo reciente? (experimento de "datos rancios")

`train` arranca en **2013**. La Clase 1 tiene el argumento exacto: *"datos de 1980 de churn de un banco no sirven para predecir hoy, porque los patrones no representan lo que pasa hoy"*. Y en la Clase 2 el profe insistió: los gustos cambian y los meses recientes "pesan más". Hay una tensión real contra otra idea de la Clase 1 (más datos → mejor, curva de aprendizaje), así que es una **pregunta empírica** — y la respondemos con la **misma CV temporal**, sin tocar `test_interno`.

Para cada ventana candidata, en cada fold expansivo filtramos el entrenamiento a los últimos *k* meses (relativo al fin de ese fold) antes de ajustar, y comparamos el AUC promedio contra usar toda la historia. La ventana ganadora define la receta de datos del modelo final.

In [ ]:
# Con los hiperparámetros ya elegidos, ¿conviene entrenar con toda la historia o solo con lo reciente?
ventanas_candidatas = [12, 24, 36, None]  # None = toda la historia
factory_arbol = lambda feats: build_tree_pipeline(feats, **mejor_params)

print("AUC CV temporal según ventana de entrenamiento (hiperparámetros fijos):")
resultados_ventana = []
for meses in ventanas_candidatas:
    filtro = (lambda df, m=meses: filtrar_reciente(df, m))
    auc_mean, _ = cv_temporal_auc(df_train, mejor_feats, factory_arbol, n_folds=N_FOLDS, train_filter=filtro)
    etiqueta = "toda la historia" if meses is None else f"últimos {meses} meses"
    resultados_ventana.append({"ventana": etiqueta, "meses": meses, "auc_cv": auc_mean})
    print(f"  {etiqueta:<18}: {auc_mean:.4f}")

vent_df = pd.DataFrame(resultados_ventana).sort_values("auc_cv", ascending=False).reset_index(drop=True)
_mejor = vent_df.loc[0, "meses"]
VENTANA_TRAIN_MESES = None if (_mejor is None or (isinstance(_mejor, float) and pd.isna(_mejor))) else int(_mejor)
print(f"\nVentana elegida: {vent_df.loc[0, 'ventana']} (AUC CV {vent_df.loc[0, 'auc_cv']:.4f})")
print("VENTANA_TRAIN_MESES =", VENTANA_TRAIN_MESES)

### Importancia de atributos del árbol elegido

La clase mostró cómo un árbol rankea variables: cada corte genera una "ganancia" (cuánto baja la impureza), y sumando las ganancias de todos los cortes que usan una variable se obtiene su importancia total. scikit-learn ya la calcula (`.feature_importances_`); la normalizamos a escala 0-100 (100 = la variable con más ganancia acumulada), igual que en clase. El árbol se ajusta con los hiperparámetros elegidos y sobre la ventana de datos elegida (`VENTANA_TRAIN_MESES`).

In [ ]:
df_train_final = filtrar_reciente(df_train, VENTANA_TRAIN_MESES)
best_tree_pipeline = build_tree_pipeline(mejor_feats, **mejor_params)
best_tree_pipeline.fit(*build_xy(df_train_final, mejor_feats))

nombres_onehot = best_tree_pipeline.named_steps["preprocessor"].get_feature_names_out()
importancias = best_tree_pipeline.named_steps["tree"].feature_importances_

imp_df = pd.DataFrame({"feature": nombres_onehot, "importancia": importancias})
imp_df = imp_df[imp_df["importancia"] > 0].sort_values("importancia", ascending=False)
imp_df["importancia_norm"] = 100 * imp_df["importancia"] / imp_df["importancia"].max()

top_n = 15
top_imp = imp_df.head(top_n).iloc[::-1].copy()
top_imp["feature"] = top_imp["feature"].str.replace("onehot__", "", regex=False)

_ventana_txt = "toda la historia" if VENTANA_TRAIN_MESES is None else f"últimos {VENTANA_TRAIN_MESES} meses"
plt.figure(figsize=(7, 5))
plt.barh(top_imp["feature"], top_imp["importancia_norm"], color=C_AZUL)
plt.xlabel("Importancia (0-100, relativa a la variable más importante)")
plt.title(f"Top {top_n} variables | depth={mejor_params['max_depth']}, "
          f"leaf={mejor_params['min_samples_leaf']} | {_ventana_txt}")
plt.tight_layout()
plt.show()

print(f"Columnas post one-hot usadas por el árbol: {len(imp_df)} de {len(nombres_onehot)}")

### Comparación final honesta: árbol vs. KNN, evaluados una única vez en `test_interno`

Ahora sí, el momento del test: lo tocamos **una sola vez**, con los dos modelos ya elegidos (no volvemos a tunear nada mirando este número). Es la única manera de responder "¿cuál de los dos es mejor?" sin el sesgo optimista de haber espiado el resultado muchas veces.

Para que sea una carrera justa, el campeón de KNN (features del conjunto C) se **reconstruye dentro de esta misma partición** y su K se elige con la **misma CV temporal** que usamos para el árbol (antes era un holdout aleatorio; ahora los dos modelos se seleccionan con idéntico régimen de validación). La muestra para entrenarlo sale solo de `train`, nunca de `test_interno`.

Por costo (KNN calcula distancia contra toda su muestra por predicción) evaluamos a ambos sobre una **submuestra estratificada de `test_interno`** de 30.000 filas; el árbol, barato de predecir, lo evaluamos además sobre el `test_interno` completo como chequeo de robustez.

Para reconstruir el campeón necesitamos el mismo pipeline de KNN de la Clase 1: one-hot encoding (la distancia euclídea no tiene sentido entre categorías) + escalado por desvío estándar sin centrar (`with_mean=False`, porque el output de `OneHotEncoder` es *sparse* y centrar lo volvería denso). El detalle completo de por qué escalar ayuda está en `tp_spotify_clase1_knn.ipynb`.

In [ ]:
def build_knn_pipeline(k, features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("onehot", OneHotEncoder(handle_unknown="ignore"), features),
        ],
        remainder="drop",
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler(with_mean=False)),
            ("knn", KNeighborsClassifier(n_neighbors=k)),
        ]
    )

In [ ]:
# --- Elegir K del campeón de KNN con la MISMA CV temporal (subsample para que sea factible) ---
KNN_SAMPLE_SIZE = 100_000   # tamaño de la muestra de entrenamiento del campeón final
KNN_CV_TRAIN = 60_000       # tope de train por fold en la CV (KNN es caro)
KNN_CV_VAL = 20_000         # tope de val por fold en la CV

candidatos_k = [101, 701, 1501]
print("Selección de K por CV temporal:")
mejor_k_final, mejor_auc_knn_val = None, -1.0
for k in candidatos_k:
    factory_knn = lambda feats, kk=k: build_knn_pipeline(kk, feats)
    auc_mean, _ = cv_temporal_auc(df_train, mejor_feats, factory_knn, n_folds=3,
                                  subsample=KNN_CV_TRAIN, val_subsample=KNN_CV_VAL)
    print(f"  K={k:>5} | ROC-AUC CV temporal = {auc_mean:.4f}")
    if auc_mean > mejor_auc_knn_val:
        mejor_k_final, mejor_auc_knn_val = k, auc_mean

print("\nK elegido para el campeón de KNN:", mejor_k_final)

# Campeón final de KNN: entrenar en una muestra de df_train (sin fuga de test_interno)
knn_sample, _ = train_test_split(
    df_train, train_size=KNN_SAMPLE_SIZE, stratify=df_train["target"], random_state=RANDOM_STATE,
)
X_knn_train, y_knn_train = build_xy(knn_sample, mejor_feats)
knn_champion = build_knn_pipeline(mejor_k_final, mejor_feats)
knn_champion.fit(X_knn_train, y_knn_train);

In [ ]:
# --- Comparación final: una única mirada a test_interno ---
test_sample, _ = train_test_split(
    df_test_interno, train_size=30_000, stratify=df_test_interno["target"], random_state=RANDOM_STATE,
)
X_test_cmp, y_test_cmp = build_xy(test_sample, mejor_feats)

auc_knn_test = roc_auc_score(y_test_cmp, knn_champion.predict_proba(X_test_cmp)[:, 1])
auc_tree_test = roc_auc_score(y_test_cmp, best_tree_pipeline.predict_proba(X_test_cmp)[:, 1])
auc_tree_test_full = roc_auc_score(y_te, best_tree_pipeline.predict_proba(X_te)[:, 1])

_ventana_txt = "toda la historia" if VENTANA_TRAIN_MESES is None else f"últimos {VENTANA_TRAIN_MESES} meses"
print("=== ROC-AUC en test_interno (tocado una única vez) ===")
print(f"KNN   (K={mejor_k_final}, muestra {KNN_SAMPLE_SIZE//1000}k de train)                 : {auc_knn_test:.4f}")
print(f"Árbol ({mejor_params}, {_ventana_txt}): {auc_tree_test:.4f}")
print(f"\nÁrbol sobre TODO test_interno ({len(df_test_interno)} filas, chequeo de robustez): {auc_tree_test_full:.4f}")

ganador = "Árbol de decisión" if auc_tree_test > auc_knn_test else "KNN"
print(f"\nGanador en test_interno: {ganador}")

### Modelo final: reentrenar con el 100% de los datos y submission

Con la receta ya elegida y validada (features, `max_depth`, `min_samples_leaf`) y ya sabiendo honestamente qué tan bien predice en `test_interno`, seguimos la misma práctica que en la Clase 1: el modelo que va a Kaggle se reentrena con el **100% de las filas de train** — ya no hace falta reservar nada, la validación cumplió su rol — porque más datos solo puede ayudar, y un árbol, a diferencia de KNN, entrena rápido incluso con el dataset completo.

In [ ]:
# Modelo final: misma receta (features + hiperparámetros + ventana de datos), reentrenada con
# el 100% de train_raw filtrado por la ventana elegida (incluye los datos más recientes, hasta 2024-08-31).
train_raw_final = filtrar_reciente(train_raw, VENTANA_TRAIN_MESES)
final_tree_pipeline = build_tree_pipeline(mejor_feats, **mejor_params)
final_tree_pipeline.fit(*build_xy(train_raw_final, mejor_feats))

# Cargar el test de la competencia y aplicar las mismas derivaciones que en train
# (mismo TOP_ARTISTAS aprendido solo de train, para no filtrar información del test)
test_raw = pd.read_csv(
    f"{DATA_DIR}/test_data.txt", sep="\t",
    usecols=[c for c in USECOLS if c != "ms_played"], dtype=DTYPES,
)
test_raw["content_type"] = np.select(
    [test_raw["spotify_track_uri"].notna(),
     test_raw["spotify_episode_uri"].notna(),
     test_raw["audiobook_uri"].notna()],
    ["track", "episode", "audiobook"],
    default="desconocido",
)
ts_test = pd.to_datetime(test_raw["ts"], utc=True, format="ISO8601")
test_raw["hora"] = ts_test.dt.hour
test_raw["dia_semana"] = ts_test.dt.dayofweek.map(dict(enumerate(DIAS)))
test_raw["artista_top"] = test_raw[col_artista].where(
    test_raw[col_artista].isin(TOP_ARTISTAS), "otros"
)

X_test_kaggle = test_raw[mejor_feats].fillna("missing").astype(str)
test_pred_proba_tree = final_tree_pipeline.predict_proba(X_test_kaggle)[:, 1]

submission_tree = pd.DataFrame({
    "obs_id": test_raw["obs_id"],
    "target": test_pred_proba_tree,
})
submission_tree.to_csv("submission_clase2_arbol.csv", index=False)
print("Guardado submission_clase2_arbol.csv —", len(submission_tree), "filas")
submission_tree.head()

### Notas honestas para el informe — Clase 2 (borrador, ir completando)

- **Qué se agregó respecto de la Clase 1:** esquema cronológico train/test_interno (85/15) con **CV temporal de ventana expansiva** dentro de train para todas las decisiones; árbol de decisión con pipeline sin escalado (los árboles no lo necesitan); selección de hiperparámetros y comparación honesta contra KNN en `test_interno` tocado una sola vez.
- **Mejoras de esta versión (todas dentro del temario de Clases 1–2):**
  1. **Validación temporal multi-ventana (ventana expansiva)** para *elegir* hiperparámetros, en lugar de un holdout cronológico único. Es la receta que la clase da para datos no-i.i.d.; promedia varias ventanas → señal menos ruidosa → `argmax` más confiable y menos "overfitting the validation set". El validation fijo de la versión anterior se eliminó: sus filas se aprovechan como desarrollo (train pasa de 70% a 85%) y su rol lo cumplen las ventanas.
  2. **Grilla CONJUNTA** sobre `max_depth × min_samples_leaf × min_samples_split × min_impurity_decrease (α)`, en vez de tunear de a un hiperparámetro (interactúan; la clase barrió 175 combinaciones en su ejemplo). `criterion` quedó **fijo en gini** para recortar cómputo — la clase lo presenta como casi equivalente a entropía; reponer en una corrida final si hay tiempo.
  3. **Experimento de datos rancios:** ¿toda la historia (desde 2013) o solo los meses recientes? Se responde con la misma CV temporal (Clase 1: "datos de 1980 no sirven para hoy"; Clase 2: los gustos cambian). La ventana ganadora (`VENTANA_TRAIN_MESES`) alimenta el modelo final.
- **Números:** completar acá con los outputs de la corrida: grilla del árbol (AUC CV temporal), ventana elegida, comparación árbol vs. KNN en `test_interno`, y experimento `user_id`. El ganador y la ventana salen del output, no están hardcodeados.
- **Punto de vista a explicitar:** el árbol minimiza impureza (Gini) internamente, pero la *selección* de hiperparámetros se hace por AUC en la CV temporal. Son dos cosas distintas a propósito (la clase: "evaluás con la métrica que te importa").
- **Cuidado con features de calendario absolutas:** `hora` y `dia_semana` son cíclicas y estables. `mes`/`año` serían time-absolute y no generalizan a la ventana futura del test — misma trampa por la que se rechazó `user_id`.
- **Limitaciones / próximos pasos:** no se re-corrió la comparación de conjuntos de features (A/B/C) específicamente para el árbol. XGBoost (adelantado por el profe como "el modelo del TP") es el salto real, pero queda fuera del temario de estas dos clases: los árboles de acá son su base conceptual.
- **Submissions:** `submission_clase2_arbol.csv`, con el 100% del train (filtrado por la ventana elegida) y la receta validada. **Ojo:** correr este notebook regenera ese archivo con la receta nueva — la versión anterior queda recuperable en git. Falta subir a Kaggle y anotar el score público.

## Mejora dentro del temario (Clases 1–2): revisar la exclusión de `user_id`

Con la Clase 2 cerrada, repasamos qué herramienta ya vista quedó sin explotar. La mejora más fuerte disponible no es una técnica nueva: es una **decisión de la Clase 1 que la Clase 2 da argumentos para revisar**. En la Clase 1 excluimos `user_id` por ser "identificador, no una característica generalizable". Revisado contra los datos crudos y contra lo que dice la clase, ese argumento no se sostiene acá:

- **El problema de los identificadores es la cardinalidad, no el nombre.** La clase lo mostró con el ejemplo del DNI: una variable con tantas categorías como filas permite una rama por observación → overfitting perfecto e inservible. `user_id` es exactamente lo contrario: **10 valores para 911.344 filas** (~90.000 observaciones por usuario; verificado sobre los archivos). La tasa de abandono de cada usuario se estima con decenas de miles de casos — es una categórica de cardinalidad bajísima, con menos categorías que `artista_top`.
- **Generaliza exactamente a donde se va a usar el modelo.** El esquema *leave-one-group-out* de la clase aplica cuando se quiere predecir en un grupo nuevo (la "provincia donde la empresa todavía no está"). No es nuestro caso: **los 10 usuarios del test de Kaggle son los mismos 10 del train** (verificado). El modelo va a predecir para las mismas personas, más adelante en el tiempo — y nuestro split cronológico replica exactamente ese escenario, que es el criterio de la clase: que validación replique el uso real.
- **El árbol es el modelo ideal para esta variable.** Cada hoja predice la proporción de clases de su región: un corte por usuario le da a cada uno su propia tasa base de abandono, y los cortes anidados captan interacciones usuario × hora, usuario × artista — la clase destacó que los árboles "captan interacciones de manera automática".

Protocolo de siempre, el que la clase usó para "¿escalar sí o no?": **una decisión nueva, un número que la valida**. Se re-corre la misma grilla conjunta **con la misma CV temporal**, con el único cambio de agregar `user_id`, y se compara contra el mejor árbol sin ella. `test_interno` no se toca en este experimento.

In [ ]:
# Diagnóstico previo, solo con train (validation/test_interno no se miran para esto):
# si la tasa de abandono varía mucho entre usuarios, la variable trae señal real.
diag_usuarios = (
    df_train.groupby("user_id")["target"]
    .agg(filas="size", tasa_abandono="mean")
    .sort_values("tasa_abandono", ascending=False)
)
print(f"Usuarios distintos en train: {len(diag_usuarios)} | tasa global de train: {df_train['target'].mean():.4f}")
diag_usuarios.round(4)

In [ ]:
# Misma grilla conjunta y misma CV temporal; la única decisión nueva es sumar user_id.
FEATURES_V2 = TODAS_LAS_FEATURES + ["user_id"]

resultados_v2 = []
for params in combos:
    factory = lambda feats, p=params: build_tree_pipeline(feats, **p)
    auc_mean, _ = cv_temporal_auc(df_train, FEATURES_V2, factory, n_folds=N_FOLDS)
    resultados_v2.append({**params, "auc_cv": auc_mean})

res_v2 = pd.DataFrame(resultados_v2).sort_values("auc_cv", ascending=False).reset_index(drop=True)
mejor_auc_v2 = float(res_v2.loc[0, "auc_cv"])

print(f"Mejor AUC CV temporal SIN user_id: {mejor_auc_arbol:.4f}")
print(f"Mejor AUC CV temporal CON user_id: {mejor_auc_v2:.4f}")
print(f"Diferencia por sumar user_id:      {mejor_auc_v2 - mejor_auc_arbol:+.4f}")
print("Config ganadora con user_id:", {k: res_v2.loc[0, k] for k in param_grid})

### Veredicto del experimento `user_id` (leer el signo de la diferencia de arriba)

- La diferencia `CON user_id − SIN user_id` decide. Con la validación **cronológica** (cada ventana entrena con el pasado y valida con su futuro inmediato), lo esperable es que **reste**, porque las tasas de abandono por usuario **no son estables en el tiempo**: el diagnóstico de arriba (celda anterior) muestra usuarios cuyo comportamiento cambia fuerte entre épocas. El árbol, goloso, elige primero esos cortes (ganancia enorme *en el pasado*) y desplaza features estables → generaliza peor al futuro.
- **Moraleja para el informe:** con un split **aleatorio** esta feature habría mostrado una mejora grande y *falsa* (mezclar épocas hace que las tasas por usuario "transfieran" entre conjuntos). El esquema cronológico + la CV temporal hacen exactamente el trabajo para el que se construyeron: rechazar señal que no generaliza. Un resultado negativo documentado es *feature engineering probado y justificado* — textual de la consigna.
- Los 10 usuarios del test de Kaggle son los mismos del train, así que la exclusión no es por falta de generalización a usuarios nuevos, sino por **inestabilidad temporal** de la señal por usuario. Completar acá con los números del output.